# CFL and dt Analysis for Static Mixer Results
This notebook scans every `cfl-time.txt` under `results/static_mixer` (including all subfolders), plots CFL and dt for each case, and reports the smallest dt values.

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np

MIN_END_TIME_SECONDS = 12.0


def find_project_root(start_dir: Path) -> Path:
    for candidate in (start_dir, *start_dir.parents):
        if (candidate / "results" / "static_mixer").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate project root. Expected results/static_mixer in the "
        "current directory or one of its parents.")


project_root = find_project_root(Path.cwd().resolve())
base_dir = project_root / "results" / "static_mixer"
pattern = re.compile(
    r"CFL:\s*([+-]?\d*\.?\d+(?:[Ee][+-]?\d+)?)\s+dt:\s*([+-]?\d*\.?\d+(?:[Ee][+-]?\d+)?)"
)
extract_pattern = re.compile(r"^\s*CFL:.*dt:.*")
case_pattern = re.compile(r"^(\d+)_re_(\d+)_pe_(\d+)$")
run_dir_pattern = re.compile(r"^run_(\d+)$")
generic_folder_tokens = {"results", "static", "mixer", "petsc"}


def moving_average(values, window_size):
    arr = np.asarray(values, dtype=float)
    if arr.size == 0:
        return np.array([]), np.array([])
    if window_size <= 0:
        raise ValueError("window_size must be positive")

    indices = np.arange(arr.size, dtype=int)
    window_starts = np.maximum(0, indices - window_size + 1)
    prefix = np.concatenate(([0.0], np.cumsum(arr, dtype=float)))
    window_sums = prefix[indices + 1] - prefix[window_starts]
    window_sizes = indices - window_starts + 1
    avg = window_sums / window_sizes
    avg_steps = indices + 1
    return avg_steps, avg


def moving_minimum(values, window_size):
    arr = np.asarray(values, dtype=float)
    if arr.size == 0:
        return np.array([]), np.array([])
    if window_size <= 0:
        raise ValueError("window_size must be positive")

    indices = np.arange(arr.size, dtype=int)
    window_starts = np.maximum(0, indices - window_size + 1)
    min_values = np.full_like(arr, np.inf)
    for i in range(arr.size):
        start = window_starts[i]
        min_values[i] = np.min(arr[start:i + 1])
    min_steps = indices + 1
    return min_steps, min_values


def folder_qualifier(file_path):
    rel_parts = file_path.relative_to(base_dir).parts
    qualifier_parts = rel_parts[:-2]
    if not qualifier_parts:
        return ""

    raw_tokens = re.split(r"[_\-]+", "_".join(qualifier_parts))
    tokens = [
        token.upper() for token in raw_tokens if token and not token.isdigit()
        and token.lower() not in generic_folder_tokens
    ]
    if tokens:
        return "-".join(tokens)

    return "_".join(qualifier_parts).upper()


def case_directories():
    return sorted(path for path in base_dir.rglob("*")
                  if path.is_dir() and case_pattern.match(path.name))


def sorted_run_directories(case_dir):
    run_dirs = []
    for path in case_dir.iterdir():
        if not path.is_dir():
            continue
        match = run_dir_pattern.match(path.name)
        if match:
            run_dirs.append((int(match.group(1)), path.name, path))

    return [
        path
        for _, _, path in sorted(run_dirs, key=lambda item: (item[0], item[1]))
    ]


def candidate_logs(folder, case_name):
    logs = []
    preferred_log = folder / f"{case_name}.log"
    if preferred_log.exists():
        logs.append(preferred_log)

    for log_file in sorted(folder.glob("*.log")):
        if preferred_log.exists() and log_file == preferred_log:
            continue
        if log_file.name == "output.log":
            continue
        logs.append(log_file)

    if not logs:
        output_log = folder / "output.log"
        if output_log.exists():
            logs.append(output_log)

    return logs


def ordered_logs_for_case(case_dir):
    ordered_logs = []
    for run_dir in sorted_run_directories(case_dir):
        ordered_logs.extend(candidate_logs(run_dir, case_dir.name))
    ordered_logs.extend(candidate_logs(case_dir, case_dir.name))

    unique_logs = []
    seen = set()
    for log_file in ordered_logs:
        if log_file in seen:
            continue
        seen.add(log_file)
        unique_logs.append(log_file)

    return unique_logs


def ensure_cfl_time(case_dir):
    cfl_path = case_dir / "cfl-time.txt"
    ordered_logs = ordered_logs_for_case(case_dir)

    refresh_reason = None
    if not cfl_path.exists():
        refresh_reason = "missing"
    elif ordered_logs:
        cfl_mtime = cfl_path.stat().st_mtime
        newest_log_mtime = max(log_file.stat().st_mtime
                               for log_file in ordered_logs)
        if newest_log_mtime > cfl_mtime:
            refresh_reason = "log_newer"

    if refresh_reason is None:
        return False, cfl_path, 0, [], "up_to_date"

    extracted_lines = []
    used_sources = []
    for log_file in ordered_logs:
        matched_lines = 0
        with log_file.open("r", errors="ignore") as handle:
            for line in handle:
                if extract_pattern.match(line):
                    extracted_lines.append(line.rstrip())
                    matched_lines += 1
        if matched_lines > 0:
            used_sources.append((log_file, matched_lines))

    if not extracted_lines:
        return False, cfl_path, 0, [], "no_matches"

    cfl_path.write_text("\n".join(extracted_lines) + "\n")
    return True, cfl_path, len(extracted_lines), used_sources, refresh_reason


refreshed_files = []
for case_dir in case_directories():
    refreshed, cfl_path, n_lines, sources, reason = ensure_cfl_time(case_dir)
    if refreshed:
        refreshed_files.append((reason, cfl_path, n_lines, sources))

if refreshed_files:
    print("Refreshed cfl-time.txt files:")
    for reason, cfl_path, n_lines, sources in refreshed_files:
        action = "created" if reason == "missing" else "updated"
        source_txt = ", ".join(f"{src.relative_to(cfl_path.parent)} ({count})"
                               for src, count in sources)
        print(f"  {cfl_path}: {action}, {n_lines} lines from {source_txt}")

else:
    print("All cfl-time.txt files are up to date.")

files = sorted(base_dir.rglob("cfl-time.txt"))
if not files:
    raise FileNotFoundError(f"No cfl-time.txt files found under {base_dir}")

parsed = {}
summary = []
case_meta = {}
skipped_short_end_time = []

for file_path in files:
    cfl_values = []
    dt_values = []

    for line in file_path.read_text().splitlines():
        match = pattern.search(line)
        if match:
            cfl_values.append(float(match.group(1)))
            dt_values.append(float(match.group(2)))

    if not dt_values:
        raise ValueError(f"No CFL/dt entries parsed in {file_path}")

    case_name = file_path.parent.name
    case_match = case_pattern.match(case_name)
    if not case_match:
        raise ValueError(f"Unexpected case folder format: {case_name}")

    resolution = int(case_match.group(1))
    reynolds = int(case_match.group(2))
    qualifier = folder_qualifier(file_path)
    case_key = f"{qualifier}/{case_name}" if qualifier else case_name

    dt_array = np.asarray(dt_values, dtype=float)
    end_time = float(np.sum(dt_array))
    if end_time < MIN_END_TIME_SECONDS:
        skipped_short_end_time.append((case_key, end_time, str(file_path)))
        continue

    step_time = np.concatenate(([0.0], np.cumsum(dt_array[:-1], dtype=float)))

    parsed[case_key] = {
        "cfl": cfl_values,
        "dt": dt_values,
        "path": file_path,
        "time": step_time,
        "end_time": end_time,
    }

    # Compute minimal dt, however we should skip the first few steps, if they
    # are monotonically increasing, as the initial step might have been too
    # small.
    first_index = 0
    for i in range(2, len(dt_values)):
        if dt_values[i] < dt_values[i - 1]:
            first_index = i - 1
            break

    min_dt = min(dt_values[first_index:])
    min_dt_step = dt_values.index(min_dt) + 1

    summary.append(
        (case_key, len(dt_values), min_dt, min_dt_step, str(file_path)))
    case_meta[case_key] = {
        "resolution": resolution,
        "re": reynolds,
        "min_dt": min_dt,
        "qualifier": qualifier,
        "end_time": end_time,
    }

if skipped_short_end_time:
    print(
        f"Excluded {len(skipped_short_end_time)} experiment(s) with end time < {MIN_END_TIME_SECONDS:.1f} s | cfl-time refreshed: {len(refreshed_files)} file(s)"
    )
    for case_key, end_time, file_name in sorted(skipped_short_end_time,
                                                key=lambda item: item[0]):
        print(f"  {case_key}: end_time={end_time:.3f}s ({file_name})")

if not parsed:
    raise ValueError(
        f"No experiments remain after applying end time filter >= {MIN_END_TIME_SECONDS:.1f} s"
    )


## Build Plots
The next cell creates one CFL and dt plot per case from the parsed data, including moving-average overlays and a folder qualifier in the title.

In [ ]:
from matplotlib.ticker import FuncFormatter

PLOT_END_TIME = MIN_END_TIME_SECONDS
MOVING_AVG_BIN_FRACTION = 0.01
MOVING_AVG_BIN_MINIMUM = 5

POINT_SIZE = 2
POINT_ALPHA = 0.30
CFL_POINT_COLOR = "#5aa2d8"
CFL_LINE_COLOR = "#0b4f8a"
DT_POINT_COLOR = "#f5a35b"
DT_LINE_COLOR = "#b34a00"
AVG_LINE_WIDTH = 2.6

dt_tick_formatter = FuncFormatter(lambda val, _: f"{val:.3e}")


def moving_average_window_size(n_steps, fraction, min_steps):
    if n_steps <= 0:
        return min_steps
    return max(min_steps, int(np.ceil(n_steps * fraction)))


def build_log_ylim(min_val, max_val, min_decades=1.0, pad_ratio=0.05):
    if min_val <= 0 or max_val <= 0:
        raise ValueError("dt must be strictly positive for log-scale plotting")

    log_min = np.log10(min_val)
    log_max = np.log10(max_val)
    log_span = max(log_max - log_min, min_decades)
    half_span = 0.5 * log_span
    log_center = 0.5 * (log_min + log_max)
    pad = pad_ratio * log_span

    return 10.0**(log_center - half_span - pad), 10.0**(log_center +
                                                        half_span + pad)


plot_cache = {}
global_dt_min = np.inf
global_dt_max = -np.inf

for case_key in sorted(parsed):
    case_data = parsed[case_key]
    time = np.asarray(case_data["time"], dtype=float)
    cfl_series = np.asarray(case_data["cfl"], dtype=float)
    dt_series = np.asarray(case_data["dt"], dtype=float)

    # Trim all times greater than 1 second
    valid_indices = time <= PLOT_END_TIME
    time = time[valid_indices]
    cfl_series = cfl_series[valid_indices]
    dt_series = dt_series[valid_indices]

    if not (time.size == cfl_series.size == dt_series.size):
        raise ValueError(
            f"Size mismatch in {case_key}: "
            f"time={time.size}, cfl={cfl_series.size}, dt={dt_series.size}")

    window_size = moving_average_window_size(
        dt_series.size,
        MOVING_AVG_BIN_FRACTION,
        MOVING_AVG_BIN_MINIMUM,
    )
    cfl_avg_steps, cfl_avg = moving_average(cfl_series, window_size)
    dt_avg_steps, dt_avg = moving_average(dt_series, window_size)

    global_dt_min = min(global_dt_min, float(np.min(dt_series)))
    global_dt_max = max(global_dt_max, float(np.max(dt_series)))

    plot_cache[case_key] = {
        "time": time,
        "cfl_series": cfl_series,
        "dt_series": dt_series,
        "cfl_avg_steps": cfl_avg_steps,
        "cfl_avg": cfl_avg,
        "dt_avg_steps": dt_avg_steps,
        "dt_avg": dt_avg,
        "window_size": window_size,
    }

global_dt_ylim = build_log_ylim(
    global_dt_min,
    global_dt_max,
    pad_ratio=0.05,
)

for case_key in sorted(parsed):
    cached = plot_cache[case_key]
    meta = case_meta[case_key]

    time = cached["time"]
    cfl_series = cached["cfl_series"]
    dt_series = cached["dt_series"]
    cfl_avg_steps = cached["cfl_avg_steps"]
    cfl_avg = cached["cfl_avg"]
    dt_avg_steps = cached["dt_avg_steps"]
    dt_avg = cached["dt_avg"]
    window_size = cached["window_size"]

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.scatter(
        time,
        cfl_series,
        color=CFL_POINT_COLOR,
        s=POINT_SIZE,
        alpha=POINT_ALPHA,
        linewidths=0,
        label="CFL (points)",
        zorder=1,
    )
    if cfl_avg.size > 0:
        cfl_avg_time = time[cfl_avg_steps - 1]
        ax.plot(
            cfl_avg_time,
            cfl_avg,
            color=CFL_LINE_COLOR,
            linewidth=AVG_LINE_WIDTH,
            label=f"CFL (Moving avg {MOVING_AVG_BIN_FRACTION*100:.1f}%)",
            zorder=3,
        )
    ax.set_ylabel("CFL", color=CFL_LINE_COLOR)
    ax.tick_params(axis="y", labelcolor=CFL_LINE_COLOR)
    ax.set_xlabel("Time")
    ax.grid(True, which="major", axis="x", alpha=0.25)

    ax2 = ax.twinx()
    ax2.scatter(
        time,
        dt_series,
        color=DT_POINT_COLOR,
        s=POINT_SIZE,
        alpha=POINT_ALPHA,
        linewidths=0,
        label="dt (points)",
        zorder=1,
    )
    if dt_avg.size > 0:
        dt_avg_time = time[dt_avg_steps - 1]
        ax2.plot(
            dt_avg_time,
            dt_avg,
            color=DT_LINE_COLOR,
            linewidth=AVG_LINE_WIDTH,
            label=f"dt (Moving avg {MOVING_AVG_BIN_FRACTION*100:.1f}%)",
            zorder=3,
        )
    ax2.set_yscale("log")
    ax2.set_ylim(*global_dt_ylim)
    ax2.set_ylabel("dt", color=DT_LINE_COLOR)
    ax2.tick_params(axis="y", labelcolor=DT_LINE_COLOR)
    ax2.yaxis.set_major_formatter(dt_tick_formatter)
    ax2.grid(True, which="both", axis="y", alpha=0.25)

    qualifier_prefix = f"{meta['qualifier']} | " if meta["qualifier"] else ""
    ax.set_title(
        f"{qualifier_prefix}Resolution: {meta['resolution']:d}, Re: {meta['re']:d}, min dt: {meta['min_dt']:.2e}"
    )
    lines = ax.get_lines() + ax2.get_lines()
    labels = [line.get_label() for line in lines]
    ax.legend(lines, labels, loc="upper right")

    plt.tight_layout()
    plt.show()


## Print Summary
The next cell prints a compact table with the minimum dt per case and reports the overall smallest dt across all processed cases.

In [ ]:
summary_sorted = sorted(summary, key=lambda item: item[0])
print("Minimum dt per file:")
print(f"{'case':40} {'entries':>8} {'min_dt':>14} {'step':>8}")
for case_key, entries, min_dt, min_dt_step, _ in summary_sorted:
    print(f"{case_key:40} {entries:8d} {min_dt:14.7e} {min_dt_step:8d}")

overall_smallest = min(summary_sorted, key=lambda item: item[2])
print(
    f"\nOverall smallest dt: {overall_smallest[2]:.7e} "
    f"(case={overall_smallest[0]}, step={overall_smallest[3]}, file={overall_smallest[4]})"
)